# Clarksons MEG VLCC position list vs Kpler

Daily Excel dumps under `broker_research/clarksons/CLARKSONS MEG VLCC POSITION *.xlsx`.

1. Parse dumps → list size / adds / removes / field changes
2. **Exit reasons** when ships leave (SUBS / HOLD / PUSHED / fixed-loaded / cancelled / unexplained)
3. Enrich via Kpler; **Fujairah** matcher basis (not Ras Tanura) vs broker open/ETA
4. PIT: live `Vessels.get` = **no history**; fixtures + matcher ladder = dated

Caches under `data/cache/` (gitignored).

In [1]:
from datetime import date, timedelta
from pathlib import Path
import sys

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from freight import kpler as kp
from freight.clarksons_meg_list import (
    annotate_exits,
    diff_consecutive_lists,
    diff_summary,
    exit_reason_summary,
    list_dynamics,
    list_snapshot_sizes,
    load_all_meg_lists,
    normalize_vessel_name,
)

pd.options.display.float_format = lambda x: f"{x:,.2f}"
CACHE = ROOT / "data" / "cache"
CLARK = ROOT / "broker_research" / "clarksons"
REFRESH = False
REFRESH_MATCHER = False  # True after weekend / new dumps
print("ROOT", ROOT)

ROOT c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\freight_dashboard


## A. Parse + list dynamics tracker

In [2]:
meg = load_all_meg_lists(CLARK)
errs = meg.attrs.get("load_errors") or []
if errs:
    display(Markdown("**Skipped / unreadable files:**\n" + "\n".join(f"- `{e}`" for e in errs)))
sizes = list_snapshot_sizes(meg)
events = diff_consecutive_lists(meg)
dyn = list_dynamics(events, sizes)
display(Markdown(
    f"Parsed **{len(meg)}** rows / **{meg['as_of'].nunique()}** as_of / "
    f"**{meg.loc[~meg['is_tbn'], 'vessel_norm'].nunique()}** unique non-TBN."
))
display(dyn)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=sizes["Date"], y=sizes["n_vessels"], name="list size", mode="lines+markers"), secondary_y=False)
if not dyn.empty:
    fig.add_trace(go.Bar(x=dyn["Date"], y=dyn["added"], name="added", marker_color="#2ca02c"), secondary_y=True)
    fig.add_trace(go.Bar(x=dyn["Date"], y=-dyn["removed"], name="removed", marker_color="#d62728"), secondary_y=True)
    fig.add_trace(go.Scatter(x=dyn["Date"], y=dyn["net"], name="net", mode="lines+markers", line=dict(dash="dot")), secondary_y=True)
fig.update_layout(title="Clarksons MEG available list — size & churn", barmode="relative", height=420)
fig.update_yaxes(title_text="vessels on list", secondary_y=False)
fig.update_yaxes(title_text="added / removed / net", secondary_y=True)
fig.show()

Parsed **2209** rows / **12** as_of / **255** unique non-TBN.

,Date,added,changed,removed,unchanged,net,n_vessels
0,2026-09-28,9,87,13,82,-4,178
1,2026-09-29,5,63,6,109,-1,177
2,2026-09-30,3,50,8,119,-5,172
3,2026-10-01,4,50,7,115,-3,169
4,2026-10-02,4,48,4,117,0,169
5,2026-10-05,53,80,10,79,43,212
6,2026-10-06,5,69,19,124,-14,198
7,2026-10-07,4,47,10,141,-6,192
8,2026-10-08,1,49,7,136,-6,186
9,2026-10-09,1,54,3,129,-2,184


## B. Exit reasons (why ships leave)

From last Clarksons row before removal + nearby Kpler fixtures. Comment tokens are **high** confidence; fixture Fully Fixed is **medium**; any nearby fixture inferred fixed is **low**.

In [3]:
config = kp.liquids_config(ROOT)
fx_start = meg["as_of"].min().date() - timedelta(days=5)
fx_end = date.today()
fixtures = kp.fixtures_vlcc_detail(config, fx_start, fx_end, from_zones=None, size=5000)
exits = annotate_exits(meg, events, fixtures)
ex_sum = exit_reason_summary(exits)
display(ex_sum)
display(Markdown(f"Total removals classified: **{len(exits)}**"))
display(exits.groupby(["exit_reason", "exit_confidence"]).size().rename("n").reset_index())

if not ex_sum.empty:
    pivot = ex_sum.pivot(index="Date", columns="exit_reason", values="n").fillna(0)
    fig_x = go.Figure()
    for col in pivot.columns:
        fig_x.add_trace(go.Bar(x=pivot.index, y=pivot[col], name=col))
    fig_x.update_layout(barmode="stack", title="Exits by reason (to_as_of)", height=400)
    fig_x.show()

display(exits[["to_as_of", "vessel", "exit_reason", "exit_confidence", "exit_source", "last_comments", "last_position"]].tail(20))

,Date,exit_reason,n
0,2026-09-28,fixed_loaded,1
1,2026-09-28,on_hold,4
2,2026-09-28,on_subs,3
3,2026-09-28,unexplained,5
4,2026-09-29,cancelled,1
5,2026-09-29,on_hold,1
6,2026-09-29,on_subs,2
7,2026-09-29,pushed,1
8,2026-09-29,unexplained,1
9,2026-09-30,fixed_loaded,1


Total removals classified: **98**

,exit_reason,exit_confidence,n
0,cancelled,medium,3
1,fixed_loaded,low,1
2,fixed_loaded,medium,3
3,on_hold,high,26
4,on_subs,high,42
5,pushed,high,1
6,unexplained,none,22


,to_as_of,vessel,exit_reason,exit_confidence,exit_source,last_comments,last_position
78,2026-10-08,C. INNOVATOR,on_hold,high,comments,HOLD,YOSU
79,2026-10-08,DESH UJAALA,on_subs,high,comments,SUBS OVER 18,SIKKA
80,2026-10-08,KALAMOS,fixed_loaded,medium,fixture_fully_fixed,None,ONSAN
81,2026-10-08,NESTOS,on_subs,high,comments,SUBS N/B,DALIAN
82,2026-10-08,PRINCESS VANYA,on_subs,high,comments,SUBS,WCIND
83,2026-10-08,SEA GEM,on_subs,high,comments,SUBS,SINGAPORE
84,2026-10-09,FPMC C MELODY,unexplained,none,none,None,NINGBO
85,2026-10-09,NEW PEARL,unexplained,none,none,None,NINGBO
86,2026-10-09,PRINCESS NATALIE,cancelled,medium,fixture_status,TBC,FUJAIRAH
87,2026-10-12,ATREBATES,unexplained,none,none,None,MALACCA


## C. Kpler enrich + Fujairah ETA ladder

Matcher destination = **`Fujairah`**. Ladder η ∈ {7,15,20,21} at 11 kn (Kpler max η=21). Compare to broker `open_date` (else `eta`).

| Field | PIT? |
|---|---|
| Live fleet status / next dest | **No** |
| Fujairah matcher ladder | **Yes** (dated) |
| Fixtures | **Yes** |

In [4]:
vessels_live = kp.vessels_vlcc_snapshot(config, cache_dir=CACHE, refresh=REFRESH)
# Re-normalize in case cache predated punctuation fix
vessels_live = vessels_live.copy()
vessels_live["vessel_norm"] = vessels_live["vessel_name"].map(normalize_vessel_name)

d0 = meg["as_of"].min().date()
d1 = min(meg["as_of"].max().date(), date.today())
ladder = kp.stitch_matcher_eta_ladder(
    config, d0, d1,
    destination=kp.FUJAIRAH_ZONE,
    etas=kp.MEG_AVAIL_ETAS,
    cache_dir=CACHE,
    refresh=REFRESH_MATCHER,
)
display(Markdown(
    f"Fujairah ladder rows: **{len(ladder)}** "
    f"(etas={kp.MEG_AVAIL_ETAS}, {d0}→{d1})."
))

# eta=7 slice for simple in_matcher flag (compat with older cells)
matcher7 = ladder.loc[ladder["matcher_eta_days"] == 7].copy() if len(ladder) else pd.DataFrame()
enriched = kp.enrich_meg_list_kpler(meg, vessels_live, fixtures, matcher7)
enriched = kp.enrich_broker_vs_kpler_eta(enriched, ladder, etas=kp.MEG_AVAIL_ETAS)

CACHE.mkdir(parents=True, exist_ok=True)
out_path = CACHE / "clarksons_meg_enriched.parquet"
enriched.to_parquet(out_path, index=False)
exits.to_parquet(CACHE / "clarksons_meg_exits.parquet", index=False)
print("wrote", out_path)

non = enriched.loc[~enriched["is_tbn"]]
match_rate = non["kpler_name_matched"].mean()
display(Markdown(
    f"Name match (non-TBN): **{100*match_rate:.1f}%**. "
    f"Fujairah ETA PIT: **{bool(non['kpler_fuj_eta_pit_available'].iloc[0])}**."
))

Fujairah ladder rows: **23923** (etas=(7, 15, 20, 21), 2026-09-25→2026-10-12).

wrote c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\freight_dashboard\data\cache\clarksons_meg_enriched.parquet


Name match (non-TBN): **99.5%**. Fujairah ETA PIT: **True**.

## D. Broker open vs Kpler Fujairah reachability

In [5]:
latest = enriched["as_of"].max()
snap = enriched.loc[(enriched["as_of"] == latest) & (~enriched["is_tbn"])].copy()

cov = pd.Series({
    "n_list": len(snap),
    "kpler_name_matched": int(snap["kpler_name_matched"].sum()),
    "in_fuj_eta7": int(snap["kpler_in_fuj_eta7"].sum()) if "kpler_in_fuj_eta7" in snap else 0,
    "in_fuj_eta15": int(snap["kpler_in_fuj_eta15"].sum()) if "kpler_in_fuj_eta15" in snap else 0,
    "in_fuj_eta21": int(snap["kpler_in_fuj_eta21"].sum()) if "kpler_in_fuj_eta21" in snap else 0,
    "can_make_broker_open": int((snap["kpler_can_make_broker_open"] == True).sum()),
    "cannot_make": int((snap["kpler_can_make_broker_open"] == False).sum()),
    "with_fixture_near": int((snap["kpler_n_fixtures_near"] > 0).sum()),
})
display(Markdown(f"### Latest `{latest.date()}`"))
display(cov.to_frame("count"))

# Overlap by eta window across as_of
eta_cols = [c for c in enriched.columns if c.startswith("kpler_in_fuj_eta")]
ov = (
    non.groupby(non["as_of"].dt.normalize())[eta_cols]
    .sum()
    .rename_axis("Date")
    .reset_index()
)
fig_o = go.Figure()
fig_o.add_trace(go.Scatter(x=sizes["Date"], y=sizes["n_vessels"], name="Clarksons list", mode="lines+markers"))
for c in eta_cols:
    fig_o.add_trace(go.Scatter(x=ov["Date"], y=ov[c], name=c.replace("kpler_in_fuj_", ""), mode="lines+markers"))
fig_o.update_layout(title="List size vs in Fujairah matcher by η window", height=420)
fig_o.show()

# Slack histogram on latest
slack = snap.loc[snap["kpler_vs_broker_eta_slack_days"].notna(), "kpler_vs_broker_eta_slack_days"]
if len(slack):
    fig_s = go.Figure(go.Histogram(x=slack, nbinsx=20, name="slack days"))
    fig_s.update_layout(
        title="Broker days-to-open − Kpler min η (latest; >0 = broker looser)",
        height=360,
    )
    fig_s.show()

show_cols = [
    "vessel", "open_date", "eta", "broker_days_to_open", "position", "comments",
    "kpler_min_eta_window", "kpler_can_make_broker_open", "kpler_vs_broker_eta_slack_days",
    "kpler_in_fuj_eta7", "kpler_in_fuj_eta15", "kpler_in_fuj_eta21",
    "kpler_imo", "kpler_next_dest", "kpler_next_eta",
]
show_cols = [c for c in show_cols if c in snap.columns]
display(Markdown("### Cannot make broker open (Kpler Fujairah)"))
display(snap.loc[snap["kpler_can_make_broker_open"] == False, show_cols].head(30))
display(Markdown("### Unmatched names"))
display(snap.loc[~snap["kpler_name_matched"], ["vessel", "dwt", "position", "open_date", "comments"]].head(20))

### Latest `2026-10-12`

,count
n_list,178
kpler_name_matched,177
in_fuj_eta7,0
in_fuj_eta15,0
in_fuj_eta21,0
can_make_broker_open,0
cannot_make,0
with_fixture_near,17


### Cannot make broker open (Kpler Fujairah)

,vessel,open_date,eta,broker_days_to_open,position,comments,kpler_min_eta_window,kpler_can_make_broker_open,kpler_vs_broker_eta_slack_days,kpler_in_fuj_eta7,kpler_in_fuj_eta15,kpler_in_fuj_eta21,kpler_imo,kpler_next_dest,kpler_next_eta


### Unmatched names

,vessel,dwt,position,open_date,comments
2079,MORINI,319.00,SINGAPORE,2026-10-16,HOLD


## E. Sanity — live vessel fields constant across as_of

In [6]:
state_cols = [c for c in ("kpler_status", "kpler_next_dest", "kpler_last_port") if c in enriched.columns]
chk = (
    enriched.loc[~enriched["is_tbn"] & enriched["kpler_name_matched"]]
    .groupby("vessel_norm")[state_cols]
    .nunique()
)
varying = chk.loc[(chk > 1).any(axis=1)]
display(Markdown(
    f"Vessels with varying live-state fields across as_of: **{len(varying)}** "
    f"(expect 0 unless snapshot refreshed mid-run)."
))
fx_by_day = (
    non.groupby(non["as_of"].dt.normalize())
    .agg(n=("vessel", "count"), with_fx=("kpler_n_fixtures_near", lambda s: (s > 0).sum()))
    .rename_axis("Date")
    .reset_index()
)
display(fx_by_day)

Vessels with varying live-state fields across as_of: **0** (expect 0 unless snapshot refreshed mid-run).

,Date,n,with_fx
0,2026-09-25,182,13
1,2026-09-28,178,8
2,2026-09-29,177,13
3,2026-09-30,172,13
4,2026-10-01,169,15
5,2026-10-02,169,19
6,2026-10-05,212,19
7,2026-10-06,198,26
8,2026-10-07,192,30
9,2026-10-08,186,37
